In [1]:
%pip install --upgrade pip
%pip install  duckdb
%pip install   numpy
%pip install  pandas
%pip install fastavro
%pip install pyspark
%pip install pyarrow matplotlib

Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 23.7 MB/s eta 0:00:0000:0100:01
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 25.9 MB/s eta 0:00:00 0:00:01
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 MB 4.2 MB/s eta 0:00:0000:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 1.4 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 1.5 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 1.7 MB/s eta 0:00:00a 0:00:01

In [2]:
#spark.stop()
from pyspark.sql import SparkSession
from pyspark import SparkConf, SparkContext
# from pyspark.sql.functions import col, count, avg, sum, min, max, desc
import sys
import time
import os
from pyspark.sql import SparkSession

# 3.5 means spark
# 2.12 means scala
# 1.10.1 - iceberg version
spark = SparkSession.builder \
    .appName("MinIO Write") \
    .master("spark://spark-master:7077") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262,org.apache.spark:spark-avro_2.12:3.5.3,org.apache.spark:spark-core_2.12:3.5.3,org.apache.spark:spark-sql_2.12:3.5.3,org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.10.1,org.apache.iceberg:iceberg-aws-bundle:1.10.1") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.iceberg.spark.SparkSessionCatalog") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.rest_prod", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.rest_prod.type", "rest") \
    .config("spark.sql.catalog.rest_prod.uri", "http://rest:8181") \
    .config("spark.sql.catalog.rest_prod.warehouse", "s3://warehouse") \
    .config("spark.sql.defaultCatalog", "rest_prod") \
    .config("spark.sql.catalog.rest_prod.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.rest_prod.s3.path.style.access", "true") \
    .config("spark.sql.catalog.rest_prod.s3.connection.ssl.enabled", "false") \
    .config("spark.sql.catalog.rest_prod.s3.endpoint", "http://minio:9000") \
    .config("spark.hadoop.mapreduce.fileoutputcommitter.algorithm.version", "2") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "2g") \
    .config("spark.executor.memoryOverhead", "2g") \
    .config("spark.driver.memoryOverhead", "1g") \
    .getOrCreate()


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
org.apache.spark#spark-avro_2.12 added as a dependency
org.apache.spark#spark-core_2.12 added as a dependency
org.apache.spark#spark-sql_2.12 added as a dependency
org.apache.iceberg#iceberg-spark-runtime-3.5_2.12 added as a dependency
org.apache.iceberg#iceberg-aws-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-dd05a29a-53aa-48ca-b522-8ce1483a7f91;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found org.apache.spark#spark-avro_2.12;3.5.3 in central
	found org.tukaani#xz;1.9 in central
	found org.apache.iceberg#iceberg-spark-runtime-3.5_2.12;1.10.1 in central
	found org.apache.iceberg

In [3]:
%%time

df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .csv("/workspace/data.csv")

df.count()

CPU times: user 24.8 ms, sys: 10.6 ms, total: 35.5 ms
Wall time: 24.5 s


7160831

In [4]:
spark.sql("CREATE DATABASE IF NOT EXISTS rest_prod.warehouse")

DataFrame[]

In [5]:
%%time
df.writeTo("warehouse.data").createOrReplace()

26/06/10 13:53:56 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


CPU times: user 20.1 ms, sys: 3.56 ms, total: 23.6 ms
Wall time: 19.4 s


In [6]:
%%time
df2 = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", "\t") \
    .csv("/workspace/data2.tsv")

df2.count()

CPU times: user 22.3 ms, sys: 5.37 ms, total: 27.7 ms
Wall time: 10.7 s


5069140

In [7]:
from pyspark.sql.functions import monotonically_increasing_id, current_timestamp

In [8]:
%%time
df2.withColumn("id", monotonically_increasing_id()).writeTo("warehouse.data2").createOrReplace()

CPU times: user 17.6 ms, sys: 11.2 ms, total: 28.8 ms
Wall time: 19.4 s


In [9]:
%ls /workspace/minio-data/

ls: cannot access '/workspace/minio-data/': No such file or directory


In [10]:
from pathlib import Path

def get_dir_size_gb(path):
    """Return directory size in GB using pathlib"""
    return sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file()) / (1024**3)

# Usage
size_gb = get_dir_size_gb('/workspace/minioData/')
print(f"{size_gb:.2f} GB")

1.35 GB


In [11]:
%%time

df2 = spark.read.table("warehouse.data2")
df2.count()

CPU times: user 2.73 ms, sys: 21 µs, total: 2.75 ms
Wall time: 322 ms


5069140

In [12]:
%%time
from pyspark.sql.functions import col
df2.filter(col("id") == 51539620450).select("id", "customer_id", "product_title", "star_rating").count()

CPU times: user 728 µs, sys: 3.49 ms, total: 4.22 ms
Wall time: 725 ms


1

In [13]:
%%time
df2.filter(col("id").isin(51539620450, 223338334409, 231928311101)) \
  .select("id", "customer_id", "star_rating", "review_headline").count()

CPU times: user 3.88 ms, sys: 1.45 ms, total: 5.33 ms
Wall time: 669 ms


3

In [14]:
%%time
df2.filter((col("id") >= 50000000000) & (col("id") <= 100000000000)) \
  .select("id", "customer_id").count()

CPU times: user 4.06 ms, sys: 0 ns, total: 4.06 ms
Wall time: 606 ms


1151928

In [15]:
from pyspark.sql.functions import col, count, avg, sum, min, max, desc

In [16]:
%%time
df2.filter((col("id") >= 50000000000) & (col("id") <= 100000000000)) \
  .select(sum(col("star_rating"))).show()

+----------------+
|sum(star_rating)|
+----------------+
|         4977220|
+----------------+

CPU times: user 2.43 ms, sys: 1.2 ms, total: 3.63 ms
Wall time: 428 ms


In [17]:
df = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("sep", "\t") \
    .csv("/workspace/data1.tsv")

from pyspark.sql.functions import monotonically_increasing_id, current_timestamp


df.withColumn("id", monotonically_increasing_id() + 10000000).writeTo("warehouse.data1").createOrReplace()

In [18]:
%%time
df1 = spark.read.table("warehouse.data1")

df1.writeTo("warehouse.data2").append()

CPU times: user 20.9 ms, sys: 7.35 ms, total: 28.3 ms
Wall time: 15.2 s


26/06/10 13:58:09 ERROR Inbox: Ignoring error
java.util.concurrent.RejectedExecutionException: Task java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask@2719615d[Not completed, task = java.util.concurrent.Executors$RunnableAdapter@6748dcc2[Wrapped task = org.apache.spark.scheduler.cluster.StandaloneSchedulerBackend$StandaloneDriverEndpoint$$anon$2@3a434ab4]] rejected from java.util.concurrent.ScheduledThreadPoolExecutor@6aae580a[Terminated, pool size = 0, active threads = 0, queued tasks = 0, completed tasks = 0]
	at java.base/java.util.concurrent.ThreadPoolExecutor$AbortPolicy.rejectedExecution(ThreadPoolExecutor.java:2055)
	at java.base/java.util.concurrent.ThreadPoolExecutor.reject(ThreadPoolExecutor.java:825)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor.delayedExecute(ScheduledThreadPoolExecutor.java:340)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor.schedule(ScheduledThreadPoolExecutor.java:562)
	at org.apache.spark.scheduler.clus

In [ ]:
%%time
df1 = spark.read.table("warehouse.data1").filter((col("id") > 0) & (col("id") < 77309418381))

df1.writeTo("warehouse.data2").append()

In [ ]:
%%time
df1 = spark.read.table("warehouse.data1").filter((col("id") > 77309418381) & (col("id") < 154618836762))

df1.writeTo("warehouse.data2").append()

In [ ]:
%%time
df1 = spark.read.table("warehouse.data1").filter((col("id") > 154618836762))

df1.writeTo("warehouse.data2").append()